# 1. 뉴스 가져오기

## 1.1. 네이트 뉴스 페이지에서 기사 가져오기

In [1]:
from wrapper.news_fetcher import NewsFetcher
from wrapper.llm_wrapper import LLM
from wrapper.api_wrapper import ApiWrapper
from tqdm import tqdm

from entity.entity import *


news = NewsFetcher()

news_list = news.fetch_news(n_pages=1)
news.save_csv()
    
news_tags = [news.tags[i][0] for i in news.tags.keys()]
news_ids = {tag: [] for tag in news_tags}

for tag in news_tags:
    news.news[tag].pop(0)

news = news.news


ModuleNotFoundError: No module named 'llama_cpp'

## 1.2. failback: csv 파일로부터 뉴스 데이터 가져오기

In [ ]:
from wrapper.news_fetcher import NewsFetcher


news = NewsFetcher().load_csv()

# 2. 뉴스 업로드

## 2.1. 업로드

In [ ]:
from wrapper.api_wrapper import ApiWrapper


api = ApiWrapper()
uploaded_news = api.upload_news(news)

## 2.2. failback: 뉴스 ID 복원

In [ ]:
from wrapper.api_wrapper import ApiWrapper
from entity.entity import *


api = ApiWrapper()
on_server = api.download_news()
uploaded_news: list[UploadedNews] = []

for tag in news:
    for n in news[tag]:
        for o in on_server:
            if o["title"] == n.title:
                uploaded_news.append(
                    UploadedNews(
                        title=n.title,
                        content=n.content,
                        image=n.image,
                        press=n.press,
                        pub_time=n.pub_time,
                        tag=n.tag,
                        url=n.url,
                        id=o["newsIdx"],
                    )
                )
                break

len(uploaded_news)

api.save_csv(uploaded_news)

## 2.3. csv 파일로부터 업로드된 뉴스 불러오기

In [ ]:
from wrapper.api_wrapper import ApiWrapper
from entity.entity import *


api = ApiWrapper()
uploaded_news = api.load_csv()

# 3. 뉴스 요약

## 3.1. 뉴스 요약 진행

In [ ]:
from wrapper.llm_wrapper import LLM
from wrapper.api_wrapper import ApiWrapper
from tqdm import tqdm

from entity.entity import *

import pickle


api = ApiWrapper()


llm = LLM(n_ctx=32768, max_tokens=1024)
summerized_news: list[SummerizedNews] = []

for news in tqdm(uploaded_news):
    llm.set_prompt(
        f"""
        [요청 사항]
        - 이 뉴스를 다음 양식을 준수하는 세 문장으로 요약해 주세요.

        [준수 사항]
        - 첫 번째 문장은 이 기사에서 다루는 핵심 사건을 설명하는 120자 내외의 완결된 문장이어야 합니다.
        - 두 번째 문장은 사건의 배경과 관련된 맥락을 설명하는 120자 내외의 완결된 문장이어야 합니다.
        - 세 번째 문장은 사건의 진행과 결과를 설명하는 120자 내외의 완결된 문장이어야 합니다.

        [참고 사항]
        - 요약하신 자료는 텍스트 임베딩을 거쳐 클러스터링 작업에 사용될 것입니다.
        - 이 뉴스는 {news.tag} 분야의 뉴스입니다.
        - 이 뉴스는 {news.pub_time} 시점에 게시되었습니다.

        [예시]
        1. 지난 23일 16시 경 광주 광산구 아파트 주차장에서 차량 4대를 들이받고 벤츠를 버린 운전자가 사건 발생 12시간 만에 경찰에 자진 출석했다.
        2. 사고 직후 운전자는 아무런 조치 없이 연락처와 벤츠를 남기고 도주했으며, 사고 12시간 40분 만에 같은 날 오후 6시쯤 경찰에 출석했다.
        3. 경찰은 A씨를 들이받은 차량을 수습하지 않은 채 도주한 혐의를 적용하여 조사하고 있으며, CCTV 등을 통해 운전 경로를 추적 수사할 방침이다.

        """
    )

    content = llm.generate(
        instruction=
        f"""
        [뉴스 제목]
        {news.title}
        [뉴스 내용]
        {news.content}
        """,
        reset_prompt=True
    )

    summerized_news.append(SummerizedNews(title=news.title, content=content, topics="", id=news.id))


with open("summerized.pkl", "wb") as f:
    pickle.dump(summerized_news, f)


def get_summerized_news(id: int) -> SummerizedNews:
    for i in summerized_news:
        if i.id == id:
            return i
    
    return None


for i in summerized_news[:5]:
    print(i.content, end="\n\n")

## 3.3. failback: 요약된 뉴스 불러오기

In [ ]:
import pickle
from entity.entity import *


with open("summerized.pkl", "rb") as f:
    summerized_news = pickle.load(f)


def get_summerized_news(id: int) -> SummerizedNews:
    for i in summerized_news:
        if i.id == id:
            return i
    
    return None

# 4. 임베딩

# 6. 클러스터링

## 6.1. DBSCAN

In [ ]:
# from wrapper.llm_wrapper import LLM
# model = LLM(embedding=True).model
from llama_cpp import Llama


del llm

MODEL_PATH = "/home/jshyeon/src/dog9/llama-3-Korean-Bllossom-8B-gguf-Q4_K_M/llama-3-Korean-Bllossom-8B-Q4_K_M.gguf"
model = Llama(model_path=MODEL_PATH, embedding=True, verbose=False)
clusters = {}

In [ ]:
import numpy as np
from tqdm import tqdm

firsts = [  ]
seconds = [ ]
thirds = [  ]

for texts in tqdm(summerized_news):
    try:
        first, second, third = list(filter(lambda x: x.strip() != '', texts.content.split('\n')))
    except:
        continue

    firsts.append(np.mean(model.embed(first, normalize=True), axis=0))
    seconds.append(np.mean(model.embed(second, normalize=True), axis=0))
    thirds.append(np.mean(model.embed(third, normalize=True), axis=0))

assert len(firsts) == len(seconds) == len(thirds)

first_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in firsts) - len(embedding)), 'constant') for embedding in firsts])
second_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in seconds) - len(embedding)), 'constant') for embedding in seconds])
third_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in thirds) - len(embedding)), 'constant') for embedding in thirds])

del model

In [ ]:
texts.content.split("\n")

In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import Normalizer


norm = Normalizer()
first_embeddings = norm.fit_transform(first_embeddings)
second_embeddings = norm.fit_transform(second_embeddings)
third_embeddings = norm.fit_transform(third_embeddings)

first_similarity = pairwise_distances(first_embeddings)
second_similarity = pairwise_distances(second_embeddings)
third_similarity = pairwise_distances(third_embeddings)

# 세 유사도의 평균값을 최종 유사도로 사용
similarity_matrix = (first_similarity + second_similarity + third_similarity) / 3
metric="euclidean"

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity


dbscan = DBSCAN(eps=0.4, min_samples=4, metric="precomputed")
clusters_ = dbscan.fit_predict(similarity_matrix)

result_string = ""

# 클러스터 결과
print(len(set(clusters_)))
clusters = {}  # 클러스터를 저장할 딕셔너리

for cluster_id in set(clusters_):
    cluster_news_ids = set()  # 중복을 피하기 위해 set 사용

    if cluster_id != -1:  # -1은 노이즈
        cluster_string = f"[Cluster ID: {cluster_id}]\n"
        result_string += cluster_string
        print(cluster_string, end="")

        for i in np.where(clusters_ == cluster_id)[0]:
            # 중복된 ID를 추가하지 않도록 set에 추가
            clustered_string = f"\t{summerized_news[i].title}\n"
            result_string += clustered_string
            cluster_news_ids.add(summerized_news[i].id)
            print(clustered_string[:-1], summerized_news[i].id)

        clusters[cluster_id] = list(cluster_news_ids)

result_string

In [ ]:
print(result_string)

# 7. 짜집기 뉴스 제작

## 7.1. (수작업) 가장 잘 군집화 된 군집 선택

In [ ]:
from wrapper.llm_wrapper import LLM

llm = LLM()

llm.__del__()
llm.__init__(n_ctx=14000)

llm.set_prompt(
    f"""
    [요구 사항]
    다음은 군집별 뉴스 기사의 제목입니다.
    제목을 보고, 가장 유사한 항목끼리 잘 묶인 군집의 Cluster ID 값을 3개만 찾아 쉼표로 구분하여 한 줄로 출력하십시오.

    [제약 사항]
    다른 수식어구 없이 해당 Cluster의 ID 값만을 말씀하십시오.
    """
)
result = llm.generate(result_string)
result

In [ ]:
top_clusters = [int(i) for i in result.split(",")]

In [ ]:
from wrapper.llm_wrapper import LLM
from entity.entity import Article
from tqdm import tqdm


llm.__del__()
llm.__init__(max_tokens=8192, temperature=0.6, top_p=0.7)


def news_creation_chain(llm: LLM, news_contents: str) -> str:
    llm.set_prompt(
        f"""
        [요청 사항]
        다음 뉴스 요약들을 종합하여 15개의 문장으로 정리해 주세요.

        [준수 사항]
        각 문장은 ~했어요, ~해요로 종결되는 300자 내외의 완결된 문장이어야 합니다.
        핵심 사건에 대한 기사 문장 5문장, 사건의 배경과 관련된 맥락에 대한 문장 5문장, 사건의 진행과 결과를 설명하는 문장 5문장으로 구성되어야 합니다.


        [참고 사항]
        하나의 뉴스 요약에서, 첫 번째 문장은 이 기사에서 다루는 핵심 사건, 두 번째 문장은 사건의 배경과 관련된 맥락, 세 번째 문장은 사건의 진행과 결과를 설명합니다.
        """
    )
    processed = llm.generate(news_contents)

    llm.set_prompt(
        f"""
        [요청 사항]
        다음 뉴스 초고를 완결된 3문단 구성의 뉴스 기사로 만들어 주세요.

        [준수 사항]
        뉴스 기사는 고등학생들이 읽을 것이에요. 친근한 말투 부탁해요.
        뉴스 기사는 3문단으로 구성되어야 해요.
        뉴스 기사의 모든 문장은 친근한 말투인 ~했어요, ~해요로 종결되어야 해요.
        뉴스 기사의 모든 문장은 300자 내외의 완결된 문장이어야 해요.

        [참고 사항]
        이 초고는 핵심 사건에 대한 기사 문장 5문장, 사건의 배경과 관련된 맥락에 대한 문장 5문장, 사건의 진행과 결과를 설명하는 문장 5문장으로 구성되어 있어요.
        """
    )
    processed = llm.generate(processed)

    llm.set_prompt(
        f"""
        [요청 사항]
        다음 문장들에 대하여, 모든 문장을 ~했어요, ~해요, ~어요로 종결되게 수정해 주세요.

        [예시]
        최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있습니다. -> 최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있어요.
        """
    )

    result = llm.generate(processed)
    return result

finals: list[Article] = []
for cluster in tqdm(top_clusters):
    llm.__del__()
    llm.__init__()
    news_contents: list = []

    for news_id in clusters[cluster]:
        target_news = get_summerized_news(news_id)
        print(target_news.title)
        news_contents.append(target_news.content)

    result = news_creation_chain(llm, '\n\n'.join(i[2:] for i in news_contents))

    llm.__del__()
    llm.__init__()

    llm.set_prompt(
        """
        [요청 사항]
        다음 Article에 대한 제목을 지어 주세요.

        [제약 사항]
        제목은 40자 내외의 간결한 문장이어야 합니다.
        """
        )

    title = llm.generate(
        f"""
        {result}
        """
    )

    finals.append(Article(title=title, content=result, news_id=clusters[cluster][:]))


In [ ]:
finals

In [ ]:
from wrapper.api_wrapper import ApiWrapper

api = ApiWrapper()

In [ ]:
import json


article_indexes = []

for article in finals:
    result = api.upload_article(article)
    print(result.status_code)
    print(json.loads(result.content.decode())["data"][0]["articleIdx"])
    article_indexes.append(json.loads(result.content.decode())["data"][0]["articleIdx"])


## 퀴즈 생성

In [ ]:
TARGET_ARTICLE = 9

In [ ]:
article = finals[TARGET_ARTICLE]

ones, twos, threes = [], [], []

for i in article.news_id:
    one, two, three = get_summerized_news(i).content.split('\n')
    ones.append(one.split('.')[1].strip())
    twos.append(two.split('.')[1].strip())
    threes.append(three.split('.')[1].strip())


llm.__del__()
llm.__init__(temperature=0.7, top_p=0.8)

llm.set_prompt(
    f"""
    [요청 사항]
    다음 뉴스 기사로부터 80자 이내의 완결된 문장을 하나 만들어 주세요.

    [제약 사항]
    문장은 완결체로 종결되어야 합니다.
    문장에 해당 뉴스 기사의 핵심 요지가 담겨있어야 합니다.

    [예시]
    미국과 일본이 체결한 안보 협력 강화 협정이 공식 발효됨에 따라, 양국 군대의 인도-태평양 지역 합동 훈련이 본격화될 전망이다.
    """
)

result = llm.generate(
    # '\n'.join(ones)
    # + '\n'.join(twos)
    # + '\n'.join(threes)
    article.content
)

result
# result = '윤석열 대통령이 박장범을 KBS 사장으로 임명하자, 더불어민주당은 "KBS를 \'김건희 방송사\'로 전락시켰다"고 강하게 비판했어요.'

In [ ]:
llm.__del__()
llm.__init__(temperature=0.6, top_p=0.9)

llm.set_prompt(
    f"""
    [지시 사항]
    아래 문장에서 첫 번째 절을 추출하십시오.
    첫 번째 절은 문장의 앞부분으로, 주어와 서술어를 포함하며, 이후 부사구, 형용사구, 종속절 등이 시작되기 전까지의 부분입니다.

    [예시]
    입력: 미국과 일본이 체결한 안보 협력 강화 협정이 공식 발효됨에 따라 양국 군대의 인도-태평양 지역 합동 훈련이 본격화될 전망이다.
    출력: 미국과 일본이 체결한 안보 협력 강화 협정이 공식 발효됨에 따라
    """
)

quiz = llm.generate(
    # '\n'.join(ones)
    # + '\n'.join(twos)
    # + '\n'.join(threes)
    result
).replace("**", "*")

quiz

In [ ]:
quiz = "한국화학연구원 연구팀이"

In [ ]:
keyword = result[len(quiz):]
print(keyword)

quiz_result = quiz + ' ' + ("_" * len(keyword))
quiz_result

In [ ]:
llm.__del__()
llm.__init__(temperature=0.7, top_p=0.7)

llm.set_prompt(
    f"""
    [요청 사항]
    이 문장을 완성시킨 결과를 출력해 주세요.

    [힌트]
    '_'의 수는 답의 글자 수입니다.
    """
    # [제약 사항]
    # 모든 문장은 ~했어요, ~해요로 종결되어야 합니다.
    # """
)

blank1 = llm.generate(quiz + keyword.split()[0])
blank2 = llm.generate(quiz + keyword.split()[0])
blank3 = llm.generate(quiz + keyword.split()[0])
blank4 = llm.generate(quiz + keyword.split()[0])
blank5 = llm.generate(quiz + keyword.split()[0])

print(blank1, '\n', blank2, '\n', blank3, '\n', blank4, '\n', blank5)

In [ ]:
#  러시아와 이란의 지원이 약화되면서 튀르키예와 사우디아라비아에 골치 아픈 상황이 예상됩니다.

blanks = [
    "주석 페로브스카이트 태양전지 생산량을 14%로 크게 향상시켰다.",
    "주석 페로브스카이트 태양전지 수율을 14%로 크게 향상시켰다.",
    "주석 페로브스카이트 태양전지 발전량을 14%로 크게 향상시켰다."
]


In [ ]:
result = api.send(
    endpoint="/quiz",
    method="POST",
    auth=True,
    data={
        "articleIdx": article_indexes[TARGET_ARTICLE],
        "quizContent": quiz_result,
        "answers": [
            {
                "answerContent": b,
                "answerYn": False
            } for b in blanks
        ]
        + [{
            "answerContent": keyword,
            "answerYn": True
        }]
    },
    query_str=False
)
print(result.status_code)

In [ ]:
print(result.content.decode())

In [ ]:
print(
    api.send(
        endpoint="/quiz",
        method="GET",
        auth=True
    ).content.decode()
)